In [ ]:
!mkdir -p /content/Bert_Senti2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!mkdir -p "/content/drive/MyDrive/Bert_Senti2"

In [ ]:
!pip install vaderSentiment

In [ ]:
import pandas as pd
import numpy as np
import string
import re


In [ ]:
!pip install fuzzywuzzy

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

In [ ]:
from imblearn.over_sampling import SMOTE

from fuzzywuzzy import fuzz
from scipy.sparse import hstack


In [ ]:
import os
import joblib

MODELS_EXIST = (
    os.path.exists("/content/drive/MyDrive/Bert_Senti2/ensemble_model.pkl") and
    os.path.exists("/content/drive/MyDrive/Bert_Senti2/tfidf_vectorizer.pkl") and
    os.path.exists("/content/drive/MyDrive/Bert_Senti2/spam_keywords.pkl") and
    os.path.exists("/content/drive/MyDrive/Bert_Senti2/spam_model_v2")
)

if MODELS_EXIST:
    print("✅ Saved models found — loading directly, skipping training.")
    ensemble      = joblib.load("/content/drive/MyDrive/Bert_Senti2/ensemble_model.pkl")
    vectorizer    = joblib.load("/content/drive/MyDrive/Bert_Senti2/tfidf_vectorizer.pkl")
    spam_keywords = joblib.load("/content/drive/MyDrive/Bert_Senti2/spam_keywords.pkl")

    from transformers import BertTokenizer, BertForSequenceClassification
    import torch
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model     = BertForSequenceClassification.from_pretrained("/content/drive/MyDrive/Bert_Senti2/spam_model_v2")
    tokenizer = BertTokenizer.from_pretrained("/content/drive/MyDrive/Bert_Senti2/spam_model_v2")
    model.to(device)
    model.eval()
    print("✅ All models loaded. Skip to prediction cells.")
else:
    print("⚠️ No saved models found — run all training cells below.")

In [ ]:
# =========================
# LOAD ALL DATASETS
# =========================

if not MODELS_EXIST:
    url1 = 'https://raw.githubusercontent.com/princebari/-SMS-Spam-Classification-on-Indian-Dataset-A-Crowdsourced-Collection-of-Hindi-and-English-Messages/main/indian_spam.csv'
    url2 = 'https://huggingface.co/datasets/thehamkercat/telegram-spam-ham/resolve/main/dataset.csv'
    url3 = 'https://huggingface.co/datasets/SetFit/enron_spam/resolve/main/enron_spam_data.csv'
    url4 = "https://raw.githubusercontent.com/War-rack/Q-Classify/main/data-en-hi-de-fr.csv"
    url5 = "https://raw.githubusercontent.com/War-rack/Q-Classify/main/indian_sms_spam_dataset_12000.csv"
    url6 = "https://raw.githubusercontent.com/War-rack/Q-Classify/main/spam_ham_dataset.csv"
    url7 = "https://raw.githubusercontent.com/War-rack/Q-Classify/main/spam.csv"
    url8 = "https://raw.githubusercontent.com/War-rack/Q-Classify/refs/heads/main/messages.csv"

    df1 = pd.read_csv(url1)
    df2 = pd.read_csv(url2)
    df3 = pd.read_csv(url3)
    df4 = pd.read_csv(url4)
    df5 = pd.read_csv(url5)
    df6 = pd.read_csv(url6)
    df7 = pd.read_csv(url7, encoding="latin1")
    df8 = pd.read_csv(url8)

    df_emails   = pd.read_csv("/content/drive/MyDrive/Bert_Senti2/emails.csv")
    df_phishing = pd.read_csv("/content/drive/MyDrive/Bert_Senti2/Phishing_Email.csv")

In [ ]:
# =========================================
# STANDARDIZE COLUMNS
# =========================================

if not MODELS_EXIST:
    def get_col(df, candidates):
        lower_map = {c.lower(): c for c in df.columns}
        for c in candidates:
            if c.lower() in lower_map:
                return lower_map[c.lower()]
        return None

    def standardize(df, label_candidates, text_candidates, fallback_label=0, fallback_text=1):
        df = df.copy()
        df.columns = df.columns.str.strip()
        lc = get_col(df, label_candidates) or df.columns[fallback_label]
        tc = get_col(df, text_candidates)  or df.columns[fallback_text]
        df = df.rename(columns={lc: "label", tc: "SMS"})
        return df[["label", "SMS"]]

    df1 = standardize(df1,
        label_candidates=["v1", "label", "class", "type", "category"],
        text_candidates =["v2", "text", "sms", "message", "content"])

    df2 = standardize(df2,
        label_candidates=["text_type", "label", "type", "class", "category", "spam"],
        text_candidates =["text", "message", "sms", "content", "body"])

    df3 = standardize(df3,
        label_candidates=["spam/ham", "label", "spam", "type", "class", "category"],
        text_candidates =["message", "text", "email text", "body", "content", "subject"])

    df4 = standardize(df4,
        label_candidates=["labels", "label", "spam", "type", "class", "category"],
        text_candidates =["text", "sms", "message", "content", "body"])

    df5 = standardize(df5,
        label_candidates=["label", "spam", "type", "class", "category"],
        text_candidates =["text", "sms", "message", "content", "body"])

    df6 = df6.drop(columns=[c for c in df6.columns if "unnamed" in c.lower() or c.lower() == "label_num"], errors="ignore")
    df6 = standardize(df6,
        label_candidates=["label", "spam", "type", "class", "category"],
        text_candidates =["text", "sms", "message", "content", "body"])

    df7 = standardize(df7,
        label_candidates=["v1", "label", "class", "type", "category", "spam"],
        text_candidates =["v2", "text", "sms", "message", "content"])

    df8 = standardize(df8,
        label_candidates=["label", "type", "class", "category", "spam"],
        text_candidates =["text", "message", "messages", "sms", "content", "body"])

    df_emails = df_emails.copy()
    df_emails.columns = df_emails.columns.str.strip()
    df_emails = df_emails.rename(columns={"text": "SMS", "spam": "label"})[["label", "SMS"]]
    df_emails["label"] = df_emails["label"].astype(int)

    df_phishing = df_phishing.copy()
    df_phishing.columns = df_phishing.columns.str.strip()
    df_phishing = df_phishing.drop(columns=[c for c in df_phishing.columns if "unnamed" in c.lower()], errors="ignore")

    print("df_phishing columns:", df_phishing.columns.tolist())

    text_col  = get_col(df_phishing, ["Email Text", "text", "email", "message", "sms", "content", "body"]) or df_phishing.columns[1]
    label_col = get_col(df_phishing, ["Email Type", "label", "type", "class", "category", "spam"])         or df_phishing.columns[0]

    df_phishing = df_phishing.rename(columns={text_col: "SMS", label_col: "label"})[["label", "SMS"]]
    df_phishing["label"] = df_phishing["label"].replace({
        "Safe Email": 0, "Phishing Email": 1,
        "safe email": 0, "phishing email": 1,
        "ham": 0, "spam": 1, "0": 0, "1": 1
    })

In [ ]:
# =========================================
# COMBINE ALL 10 DATASETS + CLEAN + LABEL ENCODE
# =========================================

if not MODELS_EXIST:
    df = pd.concat(
        [df1, df2, df3, df4, df5, df6, df7, df8, df_emails, df_phishing],
        ignore_index=True
    )
    df = df.drop_duplicates(subset=["SMS"]).reset_index(drop=True)
    print(f"Combined shape: {df.shape}")

    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = text.lower()
        text = re.sub(r"http\S+|www\S+", " ", text)
        text = re.sub(r"\S+@\S+", " ", text)
        text = re.sub(r"\d+", " ", text)
        text = text.translate(str.maketrans("", "", string.punctuation))
        text = re.sub(r"\s+", " ", text).strip()
        return text

    df["SMS"] = df["SMS"].apply(clean_text)
    df = df[df["SMS"].str.strip() != ""].reset_index(drop=True)

    df["label"] = df["label"].astype(str).str.lower().str.strip()

    label_map = {
        "ham": 0,      "0": 0,   "safe email": 0, "not spam": 0,
        "legitimate": 0, "normal": 0, "non-spam": 0, "false": 0,
        "spam": 1,     "1": 1,   "phishing email": 1, "phishing": 1,
        "smishing": 1, "fraud": 1, "scam": 1, "true": 1,
    }

    df["label"] = df["label"].map(label_map)

    unmapped = df["label"].isna().sum()
    if unmapped > 0:
        print(f"⚠️ {unmapped} rows had unmappable labels — dropped.")
        df = df.dropna(subset=["label"]).reset_index(drop=True)

    df["label"] = df["label"].astype(int)
    df = df.rename(columns={"SMS": "text"})

    print(f"✅ Final shape: {df.shape}")
    print(f"Label distribution:\n{df['label'].value_counts()}")

In [ ]:
# =========================
# FUZZY + SENTIMENT FEATURES
# =========================

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

def get_sentiment_features(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0.0, 0.0
    scores = analyzer.polarity_scores(text)
    compound = scores['compound']   # -1 to +1
    intensity = abs(compound)       # 0 to 1
    return compound, intensity

# clean_text definition (keep as is)
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

# spam_keywords (keep as is)
spam_keywords = [
    "urgent", "immediately", "expires", "limited time",
    "act now", "hurry", "last chance", "final notice",
    "win", "winner", "won", "prize", "lucky draw",
    "selected", "congratulations", "gift card", "reward",
    "free", "claim", "offer", "discount", "cashback",
    "refund", "money", "cash", "buy now", "subscribe",
    "earn", "income", "investment", "loan", "credit",
    "click", "call now", "verify", "confirm", "update",
    "submit", "download", "open", "tap here", "sign in",
    "suspended", "blocked", "expired", "unauthorized",
    "deactivated", "locked", "breach", "compromised",
    "जीत", "इनाम", "मुफ्त", "क्लिक", "अभी",
    "कॉल", "बधाई", "रुपये", "लॉटरी", "ऑफर",
    "paisa", "jeet", "inaam", "free mein", "abhi call"
]

def fuzzy_spam_score(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0.0
    keyword_score = 0
    for word in spam_keywords:
        score = fuzz.partial_ratio(word, text) / 100
        keyword_score = max(keyword_score, score)
    exclamation_score = min(text.count("!") / 5, 1)
    length_score      = min(len(text) / 200, 1)
    spam_score = (
        0.4 * keyword_score +
        0.1 * exclamation_score +
        0.1 * length_score
    )
    return spam_score

if not MODELS_EXIST:
    # Add fuzzy score
    df["fuzzy_score"] = df["text"].apply(fuzzy_spam_score)

    # ✅ ADD THIS — creates sentiment columns in df
    sentiment_results = df["text"].apply(
        lambda x: pd.Series(get_sentiment_features(x),
                           index=["sentiment_compound", "sentiment_intensity"])
    )
    df = pd.concat([df, sentiment_results], axis=1)

    print(f"✅ Fuzzy + Sentiment scores added.")
    print(df[["fuzzy_score", "sentiment_compound", "sentiment_intensity"]].describe())

In [ ]:
# =========================
# SPLIT DATA
# =========================

if not MODELS_EXIST:
    X = df["text"]
    y = df["label"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )


In [ ]:

# =========================
# TF-IDF WITH NGRAMS
# =========================

if not MODELS_EXIST:
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        max_features=5000,
        ngram_range=(2,4)
    )

    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf  = vectorizer.transform(X_test)

In [ ]:
# =========================
# ADD FUZZY FEATURE
# =========================

if not MODELS_EXIST:
    from scipy.sparse import csr_matrix

    fuzzy_train = df.loc[X_train.index, "fuzzy_score"].values.reshape(-1,1)
    fuzzy_test  = df.loc[X_test.index,  "fuzzy_score"].values.reshape(-1,1)

    # compound score: shift from [-1,+1] to [0,2] to fix MultinomialNB
    # ✅ THIS FIXES THE NEGATIVE VALUE CRASH
    senti_compound_train = (df.loc[X_train.index, "sentiment_compound"].values.reshape(-1,1) + 1)
    senti_compound_test  = (df.loc[X_test.index,  "sentiment_compound"].values.reshape(-1,1) + 1)

    # intensity is already [0,1] — no fix needed
    senti_intensity_train = df.loc[X_train.index, "sentiment_intensity"].values.reshape(-1,1)
    senti_intensity_test  = df.loc[X_test.index,  "sentiment_intensity"].values.reshape(-1,1)

    X_train_final = hstack([
        X_train_tfidf,
        csr_matrix(fuzzy_train),
        csr_matrix(senti_compound_train),
        csr_matrix(senti_intensity_train)
    ])
    X_test_final = hstack([
        X_test_tfidf,
        csr_matrix(fuzzy_test),
        csr_matrix(senti_compound_test),
        csr_matrix(senti_intensity_test)
    ])

    print(f"✅ Final feature shape: {X_train_final.shape}")
    # Should print: (n_samples, 5003)

In [ ]:
# =========================
# RandomOverSampler
# =========================

if not MODELS_EXIST:
    from imblearn.over_sampling import RandomOverSampler
    ros = RandomOverSampler(random_state=42)
    X_train_bal, y_train_bal = ros.fit_resample(X_train_final, y_train)

In [ ]:
# =========================
# MODELS
# =========================

if not MODELS_EXIST:
    log_reg = LogisticRegression(max_iter=1000)
    nb      = MultinomialNB()
    rf      = RandomForestClassifier(n_estimators=200, random_state=42)


In [ ]:

# =========================
# ENSEMBLE MODEL
# =========================

if not MODELS_EXIST:
    ensemble = VotingClassifier(
        estimators=[
            ("lr", log_reg),
            ("nb", nb),
            ("rf", rf)
        ],
        voting="soft"
    )
    ensemble.fit(X_train_bal, y_train_bal)


In [ ]:
# =========================
# MODEL EVALUATION
# =========================

if not MODELS_EXIST:
    y_pred   = ensemble.predict(X_test_final.toarray())
    accuracy = accuracy_score(y_test, y_pred)
    print("\nFinal Accuracy:", accuracy)


In [ ]:
# =========================
# PREDICTION FUNCTION
# =========================

def predict_message(message):
    message_clean = clean_text(message)

    tfidf = vectorizer.transform([message_clean])
    fuzzy = fuzzy_spam_score(message_clean)
    compound, intensity = get_sentiment_features(message_clean)

    # ✅ Same shift applied here as during training
    compound_shifted = compound + 1  # [-1,+1] → [0,2]

    from scipy.sparse import csr_matrix
    features = hstack([
        tfidf,
        csr_matrix([[fuzzy]]),
        csr_matrix([[compound_shifted]]),
        csr_matrix([[intensity]])
    ])

    pred = ensemble.predict(features.toarray())[0]
    return "Spam" if pred == 1 else "Ham"

In [ ]:
# =========================
# TEST PREDICTIONS
# =========================
""
print("\nExample Predictions\n")
print(predict_message("Final hours to get 30% off Unlimited charu, subscribe now and get one whole year to boost your knowledge, learn from top experts and master your industry’s most in-demand skills.Hurry, your offer ends today!"))

print(predict_message("Congratulations! You won a $1000 gift card. Call now!"))

print(predict_message("Hey are we meeting tomorrow for lunch?"))
print(predict_message("Please return my call"))
print(predict_message("बधाई हो! आपने 50000 रुपये जीत लिए हैं, अभी कॉल करें।"))
print(predict_message("Congratulations! आपने 10000 रुपये जीत लिए हैं, अभी क्लिक करें!"))


In [ ]:
print(predict_message("The meeting has been moved to 3 PM."))

In [ ]:
print(predict_message("आज मौसम बहुत अच्छा है।"))

In [ ]:
print(predict_message("Kal meeting hai, don't forget to bring the report."))

In [ ]:
print(predict_message("Hi Charu,kudos on completing the mock test.Great job on completing the mock test We hope you are one step closer to acing your technical interviews.We have something that can help you better prepare for your interviews. We hope you have a good time solving these questions."))

In [ ]:
print(predict_message("Dear StudentThis is to inform you that Japanese job Fair is scheduled as per the below schedule.Date - 31st Jan 2026 (Saturday)Time - 8 AMVenue - E2 Auditorium, E2 Block, Amity University, NoidaDress Code - FormalsNote - You have to be very very punctual to report on time.All the Best !!"))

In [ ]:
#-----------------------------------------------------------------#-----------------------------------------------------------------
#-----------------------------------------------------------------#-----------------------------------------------------------------
#-----------------------------------------------------------------#-----------------------------------------------------------------


In [ ]:
# =========================
# Saving model
# =========================


if not MODELS_EXIST:
    joblib.dump(ensemble,      "/content/drive/MyDrive/Bert_Senti2/ensemble_model.pkl")
    joblib.dump(vectorizer,    "/content/drive/MyDrive/Bert_Senti2/tfidf_vectorizer.pkl")
    joblib.dump(spam_keywords, "/content/drive/MyDrive/Bert_Senti2/spam_keywords.pkl")
    print("✅ Models saved successfully with sentiment features!")
else:
    print("ℹ️ Models loaded from disk.")

# FINAL PREDICTION USING BOTH BERT+SENTIMENT SCORE

In [ ]:
# =========================
#1 . Loading Bert model
# =========================

from transformers import BertTokenizer, BertForSequenceClassification
import torch

if not MODELS_EXIST:
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model     = BertForSequenceClassification.from_pretrained("/content/drive/MyDrive/Bert_Senti2/spam_model_v2")
    tokenizer = BertTokenizer.from_pretrained("/content/drive/MyDrive/Bert_Senti2/spam_model_v2")
    model.to(device)
    model.eval()

In [ ]:
# =========================
# 2. BERT SPAM PREDICTION
# =========================

def bert_result(message):

    inputs = tokenizer(
        message,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)

    spam_prob = probs[0][1].item()

    label = "Spam" if spam_prob > 0.5 else "Ham"

    return label, spam_prob


In [ ]:
# =========================
# 3. BERT RESULT FUNCTION
# =========================

def bert_result(message):
    message_clean = clean_text(message)
    inputs = tokenizer(
        message_clean,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1).squeeze()
    spam_prob = probs[1].item()
    label = "Spam" if spam_prob > 0.5 else "Ham"
    return label, spam_prob


# =========================
# 3b. ENSEMBLE RESULT
# =========================

def ensemble_result(message):
    label = predict_message(message)
    return label

In [ ]:
# =========================
# 4. FUZZY DECISION SYSTEM
# =========================

def fuzzy_decision(sentiment_label, bert_label, bert_prob):

    sentiment_score = 0.7 if sentiment_label == "Spam" else 0.3

    fuzzy_score = (0.8 * bert_prob) + (0.2 * sentiment_score)

    if fuzzy_score > 0.65:
        final = "Spam"

    elif fuzzy_score > 0.45:
        final = "Suspicious"

    else:
        final = "Ham"

    return final, fuzzy_score

In [ ]:
# =========================
# 5. FINAL PREDICTION SYSTEM
# =========================

def final_message_analysis(message):

    # Get predictions
    sentiment_label = ensemble_result(message)

    bert_label, bert_prob = bert_result(message)

    # If both models agree
    if sentiment_label == bert_label:

        final_label = bert_label
        fuzzy_score = bert_prob

    else:

        final_label, fuzzy_score = fuzzy_decision(
            sentiment_label,
            bert_label,
            bert_prob
        )

    return {

        "message": message,

        "ensemble_model": sentiment_label,

        "bert_model": bert_label,

        "bert_probability": round(bert_prob,3),

        "fuzzy_score": round(fuzzy_score,3),

        "final_verdict": final_label
    }

In [ ]:
# =========================
# 5. Test the final (BERT+SENTI) system
# =========================

print(final_message_analysis("Congratulations! You won ₹5000. Click now!"))

print(final_message_analysis("Hey are we meeting tomorrow?"))

print(final_message_analysis("URGENT! Claim your reward now"))

print(final_message_analysis("आज मौसम बहुत अच्छा है।"))

In [ ]:

print(final_message_analysis("""
Schedule Company Name - Josh Technology
Date of Online Test - 3rd Dec 2025 Reporting Time - 3:30 PM ( Late entries will not be allowed to participate)
Venue - E2 Block, 5th Floor Labs.
Important Note -It is mandatory for all students to appear for the Online Test from the designated campus labs and strictly as per the slot assigned to them.
Any student found involved in malpractice or attempting the Online Test from a remote location will be debarred from all further placement drives.
All the Best !!
Dr. Anjani Kumar Bhatnagar"""))

In [ ]:

print(final_message_analysis("""
Dear User,
We detected a login attempt from a new device in Moscow, Russia.
If this was not you, please secure your account immediately by verifying your details.
[Link: Secure My Account]Failure to verify within 24 hours will result in permanent account suspension."""))

In [ ]:

print(final_message_analysis("""
Thank you for your purchase.
 Your subscription has been renewed for $399.99. The amount will be deducted within 2 hours.
Please review the attached invoice if you did not make this transaction.[Attachment: Invoice_908234.html or .zip]"""))

In [ ]:
print(final_message_analysis("""
Hi,
Can you send me all the files of that model"""))

In [ ]:
#### TO SIMPLY DEPLOY THIS PIPELINE A USING GRADIO####
import gradio as gr

def gradio_interface(message):
    if not message or message.strip() == "":
        return {"SPAM": 0.0, "HAM": 1.0}, "No message provided."

    result = final_message_analysis(message)

    prob = result["bert_probability"]
    confidence_dict = {
        "SPAM": round(prob, 3),
        "HAM":  round(1.0 - prob, 3)
    }

    summary = (
        f"🔍 Ensemble Model  : {result['ensemble_model']}\n"
        f"🤖 BERT Model      : {result['bert_model']}\n"
        f"📊 BERT Probability: {result['bert_probability']}\n"
        f"🌀 Fuzzy Score     : {result['fuzzy_score']}\n"
        f"✅ Final Verdict   : {result['final_verdict']}"
    )

    return confidence_dict, summary

app = gr.Interface(
    fn=gradio_interface,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Type or paste your SMS/Email message here...",
        label="Message Input"
    ),
    outputs=[
        gr.Label(num_top_classes=2, label="Prediction Confidence"),
        gr.Textbox(label="Detailed Breakdown", lines=6)
    ],
    title="📬 BERT Spam Classifier",
    description="Uses fine-tuned BERT + Sentiment model + Fuzzy logic to detect spam.",
    allow_flagging="never"
)

app.launch(share=True, debug=True)


In [ ]:
# TO GENERATE PIPELINE A LINK TO ADD BELOW
import gradio as gr

def pipeline_a_predict(message):
    if not message or message.strip() == "":
        return 0.0
    result = final_message_analysis(message)
    return result["bert_probability"]

api = gr.Interface(
    fn=pipeline_a_predict,
    inputs=gr.Textbox(label="Message"),
    outputs=gr.Number(label="Spam Probability"),
    title="Pipeline A API"
)
api.launch(share=True)  # copy this generated link

In [ ]:
"""
Spam Detection System (Dual Pipeline + Fuzzy Logic)
--------------------------------------------------
- Pipeline A: BERT + Ensemble (Gradio API from Colab)
- Pipeline B: External Gradio API (VQC-QSVM)
- Fusion + Fuzzy Logic Conflict Resolver
- Safe API handling
- No fixed port (avoids crash)

Run:
    pip install gradio gradio_client
    python app.py
"""

import json
import gradio as gr
from gradio_client import Client

# ───────────────────────────────────────────── #
# CONFIG
# ───────────────────────────────────────────── #

PIPELINE_A_URL = "https://1b415c8cead4eea993.gradio.live"  # ← update when Colab restarts
PIPELINE_B_URL = "https://c07a51555a49e6efc6.gradio.live"  # ← update when Colab restarts

try:
    client_a = Client(PIPELINE_A_URL)
    print("✅ Connected to Pipeline A")
except Exception as e:
    client_a = None
    print("⚠️ Pipeline A connection failed:", e)

try:
    client_b = Client(PIPELINE_B_URL)
    print("✅ Connected to Pipeline B")
except Exception as e:
    client_b = None
    print("⚠️ Pipeline B connection failed:", e)


# ───────────────────────────────────────────── #
# PIPELINE A CALL (BERT + Ensemble)
# ───────────────────────────────────────────── #

def call_pipeline_a(message):
    if client_a is None:
        return None
    try:
        result = client_a.predict(message, api_name="/predict")
        if isinstance(result, (int, float)):
            return float(result)
        if isinstance(result, dict):
            return float(result.get("spam_prob", result.get("bert_probability", 0.5)))
        if isinstance(result, (list, tuple)):
            return float(result[0])
        return 0.5
    except Exception as e:
        print("Pipeline A ERROR:", e)
        return None


# ───────────────────────────────────────────── #
# PIPELINE B CALL (VQC-QSVM)
# ───────────────────────────────────────────── #

def call_pipeline_b(message):
    if client_b is None:
        return None
    try:
        result = client_b.predict(message, api_name="/predict")
        if isinstance(result, (int, float)):
            return float(result)
        if isinstance(result, dict):
            return float(result.get("spam_prob", 0.5))
        if isinstance(result, (list, tuple)):
            return float(result[0])
        return 0.5
    except Exception as e:
        print("Pipeline B ERROR:", e)
        return None


# ───────────────────────────────────────────── #
# FUSION ENGINE
# ───────────────────────────────────────────── #

def fuse(spam_prob_A, spam_prob_B, trust_score, alpha, w_A):
    w_B = 1 - w_A
    fused_score    = (w_A * spam_prob_A) + (w_B * spam_prob_B)
    adjusted_score = alpha * fused_score + (1 - alpha) * (1 - trust_score)
    return fused_score, adjusted_score


# ───────────────────────────────────────────── #
# FUZZY LOGIC
# ───────────────────────────────────────────── #

def fuzzy_logic(spam_prob_A, spam_prob_B, trust_score, adjusted_score):
    disagreement = abs(spam_prob_A - spam_prob_B)
    conflict = disagreement > 0.4

    if conflict:
        if adjusted_score > 0.7:
            return "Spam", conflict
        if trust_score < 0.3:
            return "Spam", conflict
        return ("Spam" if adjusted_score > 0.5 else "Not Spam"), conflict
    else:
        return ("Spam" if adjusted_score > 0.5 else "Not Spam"), conflict


# ───────────────────────────────────────────── #
# RISK SCORING
# ───────────────────────────────────────────── #

def calculate_risk_score(prob):
    if prob >= 0.85:
        return "Critical"
    elif prob >= 0.65:
        return "High"
    elif prob >= 0.35:
        return "Medium"
    else:
        return "Low"


# ───────────────────────────────────────────── #
# MAIN FUNCTION
# ───────────────────────────────────────────── #

def predict(
    message,
    manual_prob_A,   # fallback slider for Pipeline A
    manual_prob_B,   # fallback slider for Pipeline B
    trust_score,
    alpha,
    w_A,
    use_api,
):
    # ── Pipeline A (BERT + Ensemble) ──
    if use_api:
        prob_A = call_pipeline_a(message)
        if prob_A is None:
            spam_prob_A  = manual_prob_A
            api_status_A = f"⚠️ Pipeline A failed → fallback: {manual_prob_A}"
        else:
            spam_prob_A  = prob_A
            api_status_A = f"✅ Pipeline A: {spam_prob_A:.4f}"
    else:
        spam_prob_A  = manual_prob_A
        api_status_A = f"ℹ️ Manual Pipeline A: {manual_prob_A}"

    # ── Pipeline B (VQC-QSVM) ──
    if use_api:
        prob_B = call_pipeline_b(message)
        if prob_B is None:
            spam_prob_B  = manual_prob_B
            api_status_B = f"⚠️ Pipeline B failed → fallback: {manual_prob_B}"
        else:
            spam_prob_B  = prob_B
            api_status_B = f"✅ Pipeline B: {spam_prob_B:.4f}"
    else:
        spam_prob_B  = manual_prob_B
        api_status_B = f"ℹ️ Manual Pipeline B: {manual_prob_B}"

    # ── Fusion ──
    fused_score, adjusted_score = fuse(
        spam_prob_A, spam_prob_B, trust_score, alpha, w_A
    )

    # ── Fuzzy Decision ──
    final_label, conflict = fuzzy_logic(
        spam_prob_A, spam_prob_B, trust_score, adjusted_score
    )

    # ── Risk Level ──
    risk_level = calculate_risk_score(adjusted_score)

    # ── Combined API Status ──
    combined_status = f"{api_status_A}\n{api_status_B}"

    # ── Debug JSON ──
    debug = {
        "spam_prob_A":       round(spam_prob_A, 4),
        "spam_prob_B":       round(spam_prob_B, 4),
        "trust_score":       trust_score,
        "fused_score":       round(fused_score, 4),
        "adjusted_score":    round(adjusted_score, 4),
        "conflict_detected": conflict,
        "final_label":       final_label,
        "risk_level":        risk_level,
        "api_status_A":      api_status_A,
        "api_status_B":      api_status_B,
    }

    return (
        final_label,
        round(fused_score, 4),
        risk_level,
        combined_status,
        json.dumps(debug, indent=2),
    )


# ───────────────────────────────────────────── #
# GRADIO UI
# ───────────────────────────────────────────── #

with gr.Blocks(title="Spam Detection Fusion System") as demo:

    gr.Markdown("# 🛡️ Spam Detection (Dual Pipeline + Fuzzy Logic)")
    gr.Markdown("**Pipeline A** = BERT + Ensemble | **Pipeline B** = VQC-QSVM")

    message = gr.Textbox(label="Message", lines=4, placeholder="Enter message to classify...")

    use_api = gr.Checkbox(label="Use Live APIs (Pipeline A + B)", value=True)

    with gr.Row():
        manual_prob_A = gr.Slider(0, 1, 0.5, label="Manual spam_prob_A (fallback only)")
        manual_prob_B = gr.Slider(0, 1, 0.5, label="Manual spam_prob_B (fallback only)")

    with gr.Row():
        trust_score = gr.Slider(0, 1, 0.5, label="Trust Score")
        alpha       = gr.Slider(0, 1, 0.7, label="Alpha (fusion vs trust)")
        w_A         = gr.Slider(0, 1, 0.5, label="Weight A (w_B = 1 - w_A)")

    btn = gr.Button("Classify", variant="primary")

    with gr.Row():
        final_label = gr.Label(label="Final Decision")
        fused_score = gr.Number(label="Fused Score")
        risk_level  = gr.Textbox(label="Risk Level")

    api_status = gr.Textbox(label="API Status (A & B)", lines=2)
    debug      = gr.Code(label="Debug JSON", language="json")

    btn.click(
        fn=predict,
        inputs=[
            message,
            manual_prob_A,
            manual_prob_B,
            trust_score,
            alpha,
            w_A,
            use_api,
        ],
        outputs=[
            final_label,
            fused_score,
            risk_level,
            api_status,
            debug,
        ],
    )


# ───────────────────────────────────────────── #
# RUN
# ───────────────────────────────────────────── #

if __name__ == "__main__":
    demo.launch(share=True)